# CivicPulse AI — Track 2: Classical Supervised Machine Learning

Implements baseline comparison of Logistic Regression, Random Forest, and Gradient Boosting for priority prediction, and Linear Regression / Gradient Boosting for cost estimation.

In [ ]:
import sys
from pathlib import Path

ml_path = Path.cwd().parent.parent / "services" / "ml"
if str(ml_path) not in sys.path:
    sys.path.append(str(ml_path))

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor

from preprocessing.dataset_loader import generate_benchmark_synthetic_dataset, split_incident_dataset
from preprocessing.tabular_pipeline import StructuredIncidentTransformer
from evaluation.metrics import evaluate_binary_classification, evaluate_regression

# 1. Load and partition data
df = generate_benchmark_synthetic_dataset(n_samples=600, random_state=42)
train_df, val_df, test_df = split_incident_dataset(df)

# 2. Transform features
transformer = StructuredIncidentTransformer()
X_train = transformer.fit_transform(train_df)
X_test = transformer.transform(test_df)

y_train_urgent = train_df["is_urgent"].values
y_test_urgent = test_df["is_urgent"].values

y_train_cost = train_df["repair_cost"].values
y_test_cost = test_df["repair_cost"].values

# 3. Train Classification: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train_urgent)
preds_rf = rf.predict(X_test)
print("Random Forest Priority Metrics:", evaluate_binary_classification(y_test_urgent, preds_rf))

# 4. Train Regression: Gradient Boosting
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train_cost)
preds_cost = gb.predict(X_test)
print("Gradient Boosting Cost Estimation Metrics:", evaluate_regression(y_test_cost, preds_cost))